# CARP-KV GPU Validation Notebook

This notebook builds an isolated Python environment inside Colab and then runs every experiment through that environment explicitly.

That means you should be able to run the setup once per fresh Colab session without depending on notebook-kernel restarts.

Workflow:

1. Clone the repo
2. Create a Colab-local virtual environment
3. Install dependencies into that environment
4. Clone LongBench if missing
5. Run the profiling and validation scripts with the env Python

Recommended runtime:

- GPU runtime
- high-RAM if available


In [ ]:
REPO_URL = "https://github.com/SteveMama/carp-kv-research.git"
BRANCH = "main"
WORKDIR = "/content/carp-kv"
ENV_DIR = f"{WORKDIR}/.venv-colab"
PYTHON = f"{ENV_DIR}/bin/python"
PIP = f"{ENV_DIR}/bin/pip"
HF_TOKEN = ""  # optional
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
TASKS = ["qasper", "multifieldqa_en", "2wikimqa"]


In [ ]:
import os
import shutil
from pathlib import Path

if Path(WORKDIR).exists():
    shutil.rmtree(WORKDIR)

!git clone --branch {BRANCH} {REPO_URL} {WORKDIR}
%cd {WORKDIR}


In [ ]:
!python -V
!nvidia-smi || true

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF token set")

!python -m venv {ENV_DIR}
!{PIP} install -U pip setuptools wheel
!{PIP} install -U ipykernel
!{PYTHON} -V


In [ ]:
!{PIP} install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!{PIP} install -U transformers accelerate sentencepiece protobuf datasets rank_bm25 llmlingua sentence-transformers scikit-learn
!{PYTHON} - <<'PY'
import torch, transformers
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)
PY


In [ ]:
from pathlib import Path

root = Path(WORKDIR)
longbench_repo = root / "LongBenchRepo"
if not longbench_repo.exists():
    !git clone https://github.com/THUDM/LongBench.git {longbench_repo}

print("LongBench repo:", longbench_repo)
!ls -R {longbench_repo} | sed -n "1,80p"


## 1. Offline Qwen KV Profiling

This reproduces the offline codec-quality experiments and refreshes the selector training artifacts.


In [ ]:
!{PYTHON} profile_qwen_kv_polar.py \
  --model-name {MODEL_NAME} \
  --tasks qasper multifieldqa_en 2wikimqa \
  --samples-per-task 2 \
  --max-context-tokens 1024 \
  --max-vectors-per-item 4096


## 2. Single-Step Cache-Path Validation

Run the two exact-head thresholds that were important locally.


In [ ]:
!{PYTHON} carp_cache_eval.py \
  --tasks qasper multifieldqa_en 2wikimqa \
  --samples-per-task 5 \
  --full-max-context-tokens 512 \
  --exact-head-risk-threshold 0.8 \
  --output results/carp_cache_eval_thr08_gpu.json

!{PYTHON} carp_cache_eval.py \
  --tasks qasper multifieldqa_en 2wikimqa \
  --samples-per-task 5 \
  --full-max-context-tokens 512 \
  --exact-head-risk-threshold 0.7 \
  --output results/carp_cache_eval_thr07_gpu.json


## 3. Multi-Step Decode Stability

Current best local operating point:

- exact-head threshold = `0.7`
- entropy fallback threshold = `0.30`


In [ ]:
!{PYTHON} carp_multistep_eval.py \
  --tasks qasper multifieldqa_en 2wikimqa \
  --samples-per-task 5 \
  --decode-steps 8 \
  --exact-head-risk-threshold 0.7 \
  --entropy-fallback-threshold 0.30 \
  --output results/carp_multistep_eval_thr07_entropy03_gpu.json


## 4. Optional Llama Validation

Once Qwen runs are stable, switch to Llama for the publishable comparison.

Replace `MODEL_NAME` above and rerun the same sequence if your Colab GPU has enough memory.


In [ ]:
import json
from pathlib import Path

for path in [
    Path("results/qwen_polar_profile.json"),
    Path("results/carp_cache_eval_thr07_gpu.json"),
    Path("results/carp_multistep_eval_thr07_entropy03_gpu.json"),
]:
    if path.exists():
        data = json.loads(path.read_text())
        print("=" * 80)
        print(path)
        print(json.dumps(data, indent=2)[:4000])
    else:
        print("missing:", path)


## 5. Notes

- Every script in this notebook runs through `"$PYTHON"`, the Colab-local virtual environment.
- If you reconnect to a fresh runtime, rerun the setup cells first.
- The 8-step decode test is a stability proxy, not a full LongBench answer-quality benchmark.
- For final task-level claims, run full-answer generation and score against the real LongBench answers.
- The main hypothesis to validate on GPU is that larger models trigger exact-step fallback much less often than the local 0.5B Qwen path.
